In [22]:
import pyomo.environ as pyo
from pyomo.gdp import Disjunction, Disjunct
import json
import os

In [23]:
json ={
    "jobs": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    "processing_time": {
        "1": 34, "2": 38, "3": 39, "4": 20, "5": 47, "6": 47, "7": 14, "8": 13,
        "9": 28, "10": 20, "11": 32, "12": 25, "13": 16, "14": 35, "15": 11
    },
    "release_time": {
        "1": 64, "2": 195, "3": 200, "4": 192, "5": 135, "6": 189, "7": 82, "8": 104,
        "9": 123, "10": 29, "11": 43, "12": 49, "13": 156, "14": 11, "15": 52
    },
    "due_time": {
        "1": 302, "2": 461, "3": 473, "4": 332, "5": 464, "6": 518, "7": 180, "8": 195,
        "9": 319, "10": 169, "11": 267, "12": 224, "13": 268, "14": 256, "15": 129
    }
}

In [41]:
m = pyo.ConcreteModel()

m.I = pyo.Set(initialize=json["jobs"])
m.p = pyo.Param(m.I, initialize={int(k): v for k, v in json["processing_time"].items()})
m.r = pyo.Param(m.I, initialize={int(k): v for k, v in json["release_time"].items()})
m.d = pyo.Param(m.I, initialize={int(k): v for k, v in json["due_time"].items()})

# Immediate precedence Concepts
m.x = pyo.Var(m.I, within=pyo.NonNegativeReals)

# first job disjuncts
def first_job_disjunct_rule(disjunct, i):
    m = disjunct.model()
    disjunct.cons=pyo.ConstraintList()
    # xi+ pi <= xj for all i not equal to j
    for j in m.I:
        if i != j:
            disjunct.cons.add(m.x[i] + m.p[i] <= m.x[j])
m.first_job_disjunct = Disjunct(m.I, rule=first_job_disjunct_rule)

# last job disjuncts
def last_job_disjunct_rule(disjunct, i):
    m = disjunct.model()
    disjunct.cons=pyo.ConstraintList()
    # xj + pj <= xi for all i not
    for j in m.I:
        if i != j:
            disjunct.cons.add(m.x[j] + m.p[j] <= m.x[i])
m.last_job_disjunct = Disjunct(m.I, rule=last_job_disjunct_rule)

def first_job_disjunction_rule(m):
# Return a list of first-job disjuncts, one per job
        return [m.first_job_disjunct[i] for i in m.I]
m.first_job_disjunction = Disjunction(rule=first_job_disjunction_rule)

def last_job_disjunction_rule(m):
# Return a list of last-job disjuncts, one per job
    return [m.last_job_disjunct[i] for i in m.I]
m.last_job_disjunction = Disjunction(rule=last_job_disjunction_rule)

# Logic Expression
def logic_expression_rule(m, i):
    return pyo.lnot(pyo.land(m.first_job_disjunct[i].indicator_var, m.last_job_disjunct[i].indicator_var))
m.logic_expression = pyo.LogicalConstraint(m.I, rule=logic_expression_rule)

# Immediate precedence disjuncts
def immediate_precedence_disjunct_rule(disjunct, i, j):
    m = disjunct.model()
    if i == j:
        disjunct.deactivate() # Deactivate the disjunct if i == j
    else:
        disjunct.cons = pyo.Constraint(expr=m.x[i] + m.p[i] <= m.x[j])
m.immediate_precedence_disjunct = Disjunct(m.I, m.I, rule=immediate_precedence_disjunct_rule)

# Successor Disjunction
def successor_disjunction_rule(m, i):
    return [m.immediate_precedence_disjunct[i, j] for j in m.I if j != i] + [m.last_job_disjunct[i]]
m.SuccessorDisjunction = Disjunction(m.I, rule=successor_disjunction_rule)

# Predecessor Disjunction
def predecessor_disjunction_rule(m, i):
    return [m.immediate_precedence_disjunct[j, i] for j in m.I if j != i] + [m.first_job_disjunct[i]]
m.PredecessorDisjunction = Disjunction(m.I, rule=predecessor_disjunction_rule)